# 00 — Setup & Smoke Test
**Objective:** prove Spark 4.2 runs on the standalone cluster and the ObjectScale PoC environment is configured.

**Customer sees:** versions, cluster state, and a real Spark job.
**TC:** TC0.1

## Step 1 — Start the session
Loads `.env`, builds a SparkSession on the standalone master, prints the run ID, the identities the session uses, masked credentials and versions read from the running components.

In [ ]:
import time
import pandas as pd
from poc import config, results, viz
import poc.spark_helpers as sh

spark = sh.build_session("obs-poc")
viz.table(pd.DataFrame(sh.versions(spark).items(), columns=["component", "version"]))
print("run id:", results.run_id())
print("session identities:", sh.identities(spark))
print("users:", config.masked_users())

**Presenter notes** (hidden)

Point out: no secrets printed — all keys masked. Versions are read live (ObjectScale from the management API, Iceberg from the loaded runtime). Spark UI: see the table.

## Step 2 — Cluster smoke job (TC0.1)
`spark.range(10M).count()` runs a real distributed job on the worker and returns the count — proves the cluster computes.

In [ ]:
t0 = time.time(); n = spark.range(10_000_000).count(); dt = time.time() - t0
res = results.write("TC0.1", "spark.range(10M).count()", n == 10_000_000,
                    {"count": n, "seconds": round(dt, 1)})
viz.table(pd.DataFrame([{"count": n, "seconds": round(dt, 1)}]))
viz.banner(res["status"], "TC0.1 smoke", f"{n:,} rows in {dt:.1f}s")
spark.stop()

**Presenter notes** (hidden)

If this is slow, the worker may still be registering — wait a few seconds.